In [ ]:
import numpy as np
import pandas as pd
import ast
import nltk

In [ ]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [ ]:
movies.head(1)

In [ ]:
movies.info()

In [ ]:
credits.head(1)

### Now our job is marge this dataset

In [ ]:
movies = movies.merge(credits,on='title')

In [ ]:
movies.sample(10)

### what kinds of column i need

In [ ]:
# geners
# id
# title
# overview
# keywords
# cast
# crew

movies = movies[['id','title','overview','genres','keywords','cast','crew']]
movies.head(10)

In [ ]:
movies.isnull().sum()

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies.duplicated().sum()

In [ ]:
movies.iloc[0].genres

In [ ]:
def convert(obj):
  L = []
  for i in ast.literal_eval (obj):
    L.append(i['name'])
  return L

In [ ]:
movies['genres'] = movies['genres'].apply(convert)

In [ ]:
movies.head(10)

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
def convert3(obj):
  L = []
  counter = 0
  for i in ast.literal_eval (obj):
    if counter != 3:
       L.append(i['name'])
       counter += 1
    else:
      break
  return L

In [ ]:
movies['cast'] = movies['cast'].apply(convert3)

In [ ]:
def fetch_director(obj):
  L = []

  for i in ast.literal_eval (obj):
    if i['job'] == 'Director':
       L.append(i['name'])
       break


  return L

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies.head(1)

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [ ]:
movies.head(1)

In [ ]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['cast']

In [ ]:
movies.head()

In [ ]:
new_df = movies[['id','title','tags']]
new_df

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

In [ ]:
new_df['tags'][0]

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

# text vectorization

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
def stem(text):
  y = []
  for i in text.split():
    y.append(ps.stem(i))
  return " ".join(y)

In [ ]:
ps.stem('loving')

In [ ]:
new_df['tags'] = new_df['tags'].apply(stem)

In [ ]:
cv = CountVectorizer(max_features=5000,stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vectors[0]

In [ ]:
cv.get_feature_names_out()

# distance maping

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(vectors)

In [ ]:
sorted(list(enumerate(similarity[0])),reverse=True,key = lambda x:x[1])[1:6]

# Recommendation Process

In [ ]:
def recommend(movie):
  movie_index = new_df[new_df['title']==movie].index[0]
  distances = similarity[movie_index]
  movies_list = sorted(list(enumerate(distances)),reverse=True,key = lambda x:x[1])[1:6]

  for i in movies_list:
    print(new_df.iloc[i[0]].title)



In [ ]:
recommend('Spider-Man')

# Time  to connected with website

In [ ]:
import pickle

In [ ]:
pickle.dump(new_df.to_dict(),open('movies.dict.pkl','wb'))

In [ ]:
new_df['title'].values

In [ ]:
pickle.dump(similarity,open('similarity.pkl','wb'))

In [ ]:
from google.colab import files
files.download('similarity.pkl')

In [ ]:
new_df